In [1]:
from __future__ import annotations
from typing import Iterable, Callable

from tqdm import tqdm
import concurrent.futures

import random
import numpy as np
import pandas as pd
from math import log
import networkx as nx
from scipy.stats import pearsonr
from itertools import combinations
from sklearn.metrics import adjusted_rand_score

from freyrelab.regnets import regnet
from freyrelab.regnets.abasy import Abasy
from freyrelab.regnets.regnet import RegNet
from freyrelab.nets import models, dissimilarity

from multiprocessing import cpu_count

from dvalue_paral import run_parallel, run_d	

In [3]:
workers = cpu_count()-2
workers

14

In [4]:
abasy = Abasy(db='abasy_internal', expire_cache=True)
regnet_ids = abasy.select_regnets(nr_strong_wa=True)    # select all regnets without redundancy, keep the strong one if available
regnets = abasy.regnet(regnet_ids)                      # get the regnets {regnet_id: RegNet}


seed = 42
random.seed(seed)
random_graph = {}
hm_seed_size = 3

for net_id, G in regnets.items():
    n = G.number_of_nodes()
    m = G.number_of_edges()
    random_graph[f'BA_{net_id}'] = models.barabasi_albert_graph(n)
    random_graph[f'SF_{net_id}'] = nx.DiGraph(nx.scale_free_graph(n, seed=seed))
    random_graph[f'ER_{net_id}'] = nx.erdos_renyi_graph(n, m/(n*(n-1)), seed=seed, directed=True)
    random_graph[f'HM_{net_id}'] = models.hierarchical_modular_graph(int(log(n, hm_seed_size))-1, m=hm_seed_size)

networks = {**regnets, **random_graph}
networks = {name: RegNet(G) if not isinstance(G, (nx.DiGraph,RegNet)) else G for name, G in networks.items()}

In [5]:
# # networks = {str(i):nx.erdos_renyi_graph(100, 0.2) for i in range(100)}
network_names = list(networks.keys())
combs = list(combinations(network_names, 2))
data = [
    combs,
    [(networks[n1], networks[n2]) for n1, n2 in combs],
]
results = run_parallel(run_d, data, workers)

  0%|          | 0/2 [00:00<?, ?it/s]

Error: cannot pickle 'dict_keys' object


24it [00:10,  4.21it/s]                      

Error: cannot pickle 'dict_keys' object


197it [09:01,  9.82s/it]

Error: cannot pickle 'dict_keys' object


229it [12:02,  2.99s/it]

Error: cannot pickle 'dict_keys' object


272it [15:06,  7.46s/it]

In [ ]:
corr_df = pd.DataFrame(index=network_names, columns=network_names)

for (net1, net2), d in results.items():
    corr_df.loc[net1, net2] = d
    corr_df.loc[net2, net1] = d

corr_df = corr_df.astype(float).fillna(0)
corr_df.to_csv("dissimilarity_abasy.csv")

In [12]:
networks = {str(i):nx.erdos_renyi_graph(100, 0.2) for i in range(100)}

In [13]:
network_names = networks.keys()
corr_df = pd.DataFrame(index=network_names, columns=network_names)

for net1, net2 in combinations(network_names, 2):
    d = dissimilarity.graph_dissimilarity(networks[net1], networks[net2])
    corr_df.loc[net1, net2] = d
    corr_df.loc[net2, net1] = d

corr_df = corr_df.astype(float).fillna(0)
#corr_df.to_csv("dissimilarity.csv")

ValueError: math domain error